In [10]:
import sys
if "vision_env" not in sys.executable:
    print("/n环境配置错误!!!/n")
    print(sys.executable)
else:
    print("环境配置正常")

环境配置正常


In [11]:
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt 
from matplotlib import rcParams # 字体配置,支持中文
rcParams['font.family'] = 'SimHei'
rcParams['axes.unicode_minus'] = False  # 解决负号不显示的问题

import cv2

print("opencv:", cv2.__version__)
print("numpy:", np.__version__)

opencv: 4.13.0
numpy: 2.2.5


<font color = #000000 >

## 10. 案例2:OCR文字识别
</font>


In [12]:
# 前期准备

picture = './Pictures/'

# 1. 图像读取
def read_img(path, flag=cv2.IMREAD_COLOR):
    img = cv2.imread(path, flag)
    if img is None:
        raise FileNotFoundError(f"图像读取失败: {path}")
    else:
        print('图像读取成功, shape:' , img.shape)
    return img

# 2. 定义图像展示函数
def img_show(img, title="image"):
    try:
        # 1 判断 None
        if img is None:
            print("图像是None")
            return
        
        # 2 判断类型
        if not isinstance(img, np.ndarray):
            print("输入不是 numpy.ndarray")
            print("type =", type(img))
            return
        
        # 3 判断 shape
        if img.size == 0:
            print("图像为空")
            return
        
        # 4 打印基本信息
        print("shape =", img.shape)
        print("dtype =", img.dtype)

        # 5 判断通道
        if len(img.shape) == 2:
            # 灰度图
            show_img = img
        
        elif len(img.shape) == 3:
            if img.shape[2] == 3:
                show_img = img
            elif img.shape[2] == 4:
                # RGBA → RGB
                show_img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
            else:
                print(" 不支持的通道数:", img.shape)
                return
        else:
            print(" 不支持的图像维度:", img.shape)
            return

        # 6 显示图像
        cv2.imshow(title, show_img)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

    except Exception as e:
        print(" 图像显示失败:", e)

# 3. 定义图像resize函数
def my_resize(image, width=None, height=None, inter=cv2.INTER_AREA):
    dim = None
    (h, w) = image.shape[:2]
    if width is None and height is None:
        return image
    if width is None:
        if (height is not None):
            r = height / float(h)
            dim = (int(w * r), height)
    else:
        r = width / float(w)
        dim = (width, int(h * r))
    resized = cv2.resize(image, dim, interpolation=inter)
    print(f'图像size配置完成,原{(h, w)}-> 现{dim}')
    return resized

<font color = #000000>

### 10-1 图像轮廓检测


</font>

In [29]:
# 1. 先读取图像
img = read_img(picture + 'receipt.jpg')
# 2. 图片大小更改
ratio = img.shape[0] / 500.0
orig = img.copy()   # 原图
img = my_resize(img , height=500)   
# 3. 高斯去噪音+灰度+二值化处理
img = cv2.GaussianBlur(img, (5, 5), 1)  # 缩略模糊图
thresh = cv2.cvtColor(img , cv2.COLOR_BGR2GRAY)
_ , thresh = cv2.threshold(thresh , 127 , 255 , cv2.THRESH_BINARY)  # 缩略模糊二值化图

图像读取成功, shape: (2448, 3264, 3)
图像size配置完成,原(2448, 3264)-> 现(666, 500)


In [ ]:
# 4. 轮廓检测

# 4-1 背景图像配置
img_bg = cv2.cvtColor(img.copy() , cv2.COLOR_BGR2RGB)
thresh_bg = cv2.cvtColor(thresh.copy() , cv2.COLOR_GRAY2RGB)

# 4-2 边缘检测,发现进行了边缘检测得到的边界增多,所以算了
# thresh = cv2.Canny(thresh , 75  , 200)
# img_show(thresh)

# 4-3 轮廓检测
contours , _ = cv2.findContours(thresh , cv2.RETR_TREE , cv2.CHAIN_APPROX_NONE)
# ret = cv2.drawContours(thresh_bg.copy() , contours , -1 , (255,0,0) , 2)
# img_show(ret)

# 4-4 轮廓筛选:选择最大的轮廓
cnt = sorted(contours , key=lambda x: cv2.contourArea(x) , reverse=True)
# ret = cv2.drawContours(thresh_bg.copy() , cnt , 0 , (255,0,0) , 2)
# img_show(ret)
edge = cnt[0]
# ret = cv2.drawContours(thresh_bg.copy() , [edge] , 0 , (255,0,0) , 2)
# img_show(ret)

# 4-5 使用轮廓近似,读取轮廓的角点
epsilon = 0.01 * cv2.arcLength(edge , True)
edge  = cv2.approxPolyDP(edge , epsilon , True)
print(f"读取到了edge:\n{edge}")

# res = cv2.drawContours( thresh_bg , [edge] , -1 , (0,0,255) , 2)
# img_show(res)

读取到了edge:
[[[463 111]]

 [[115 137]]

 [[148 375]]

 [[473 323]]]
<class 'numpy.ndarray'>


<font color = #000000>

### 10-2 图像仿射变换


</font>

In [28]:
# 处理点坐标，返回rect使其顺序为左上，右上，右下，左下
def order_points(pts):
    # 一共4个坐标点
    rect = np.zeros((4, 2), dtype="float32")
 
    # 计算左上，右下  左上的x和y都是最小的 右下的x和y都是最大的 
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
 
    # 计算右上和左下  np.diff后一项减前一项
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

# 把原图转换为“铺平 铺满”后的图片
def four_point_transform(image, pts):
    # 获取输入坐标点
    rect = order_points(pts)
    tl, tr, br, bl = rect
 
    # 两点间距离公式计算输入的w和h值
    widthTop = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
    widthBot = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
    # 要最大的 看着方便 下同 
    maxWidth = max(int(widthTop), int(widthBot))
 
    heightRight = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
    heightLeft = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
    maxHeight = max(int(heightRight), int(heightLeft))
 
    # 变换后对应坐标位置
    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]], dtype="float32")   
    
    # 计算变换矩阵
    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))
      
    # 返回变换后结果
    return warped


In [ ]:
# 透视变换  记得乘以比例，之前为了方便观察统一过大小。
warped = four_point_transform(orig, edge.reshape(4,2) * ratio)
img_show(my_resize(warped , height=500))    # 缩小尺寸只是为了电脑看得完整而已

图像size配置完成,原(1176, 1708)-> 现(726, 500)
shape = (500, 726, 3)
dtype = uint8


<font color = #000000>

### 10-3 OCR读取


</font>

In [ ]:
# 略